In [ ]:
import cv2
import yagmail
from geopy.geocoders import Nominatim
import datetime
import os

# Setup Geopy geolocator
geolocator = Nominatim(user_agent="geoapiExercises")

def capture_image(save_path="captured_image.jpg"):
    """
    Captures an image from the default camera and saves it.
    """
    cap = cv2.VideoCapture(0)  # 0 = default camera

    if not cap.isOpened():
        print("❌ Cannot open camera")
        return None

    ret, frame = cap.read()
    if ret:
        cv2.imwrite(save_path, frame)
        print("✅ Image captured successfully!")
    else:
        print("❌ Failed to capture image")
        save_path = None

    cap.release()
    cv2.destroyAllWindows()
    return save_path

def email_generate_with_image(t1, t2, latitude, longitude, image_path):
    """
    Sends an email with geolocation info, time details, and an attached image.
    """
    try:
        # Calculate time difference
        time_difference = (t2 - t1)
        hours_passed = time_difference.total_seconds() / 3600 if not isinstance(time_difference, (int, float)) else time_difference

        # Get address from latitude and longitude
        location = geolocator.reverse(f"{latitude}, {longitude}", exactly_one=True)
        address = location.address if location else "Unknown Location"

        # Google Maps Link
        google_maps_link = f"https://www.google.com/maps/search/?api=1&query={latitude},{longitude}"

        # Email credentials
        from_address = "trashtrackerproject@gmail.com"   # Sender Gmail
        to_address = "shreya06lg@gmail.com"               # Receiver Gmail
        app_password = "kecvuuderioxpsqk"              # App password for Gmail

        subject = "🚨 Real-Time Scrap Detection Alert!"
        case_title = "Attention Required!"

        # Email HTML Body
        html_content = f"""
        <html>
            <body>
                <h1 style="color:red;">{case_title}</h1>
                <h2>📍 Address:</h2>
                <p>{address}</p>
                <h3>🕒 Scrap Detected Time: {t1.strftime('%Y-%m-%d %H:%M:%S')}</h3>
                <h3>📍 View Location:</h3>
                <a href="{google_maps_link}" style="font-size:18px; color:blue;">Open in Google Maps</a><br><br>
                <p>Attached below is a live captured image of the event:</p>
            </body>
        </html>
        """

        # Connect to Gmail using Yagmail
        yag = yagmail.SMTP(from_address, app_password)

        # Send the email
        yag.send(
            to=to_address,
            subject=subject,
            contents=[html_content, yagmail.inline(image_path)]
        )

        print("✅ Email with image sent successfully!")

    except Exception as e:
        print(f"❌ Error sending email: {e}")

# ---------- MAIN RUNNING SECTION ------------

if __name__ == "__main__":
    # Capture current time
    current_time = datetime.datetime.now()

    # Example coordinates (can be from GPS device or manually set)
    latitude = 12.957771112514944
    longitude = 77.58046134865555

    # Capture image from webcam
    captured_image_path = capture_image()

    # If image captured successfully, send email
    if captured_image_path:
        email_generate_with_image(current_time, current_time + datetime.timedelta(hours=1), latitude, longitude, captured_image_path)

        # Cleanup: Delete the captured image after sending
        try:
            os.remove(captured_image_path)
            print("🧹 Temporary image deleted.")
        except Exception as e:
            print(f"⚠️ Could not delete temporary image: {e}")


Done
